# Conversation memory with LangGraph

In Module 1 you kept a Python list of messages and sent it on every call. Here the graph keeps the messages in its state, and a checkpointer saves that state between calls.

Open this notebook from its `Code` folder and select the course Python kernel.

## Setup
Load the course key and create the LangChain client.

In [ ]:
from pathlib import Path
import yaml

key_file = Path("../../../llm_provider_api_key.yaml")
api_key = yaml.safe_load(key_file.read_text())["ollama"]["api_key"]


from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="gpt-oss:120b",
    base_url="https://ollama.com",
    client_kwargs={"headers": {"Authorization": "Bearer " + api_key}},
)

## 1. Messages in state
By default, a node update replaces a key's value. A **reducer** combines the update with the current value.

`add_messages` appends new messages and updates messages with the same ID. It combines messages within a run; retaining them across calls requires the checkpointer in section 3.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

class ChatState(TypedDict):
    messages: Annotated[list, add_messages]

This node does not call a model: it repeats the last message. It returns only the new message; the reducer appends it.

In [ ]:
def echo(state: ChatState):
    last = state["messages"][-1].content
    return {"messages": [AIMessage("You said: " + last)]}

builder = StateGraph(ChatState)
builder.add_node("echo", echo)
builder.add_edge(START, "echo")
builder.add_edge("echo", END)
echo_graph = builder.compile()

result = echo_graph.invoke({"messages": [HumanMessage("Hello")]})
for message in result["messages"]:
    message.pretty_print()

LangGraph also provides `MessagesState`, a ready-made state with the same `messages` key and reducer. We use it from now on.

## 2. A chatbot node
The node sends the system instruction and the messages in state to the model, then returns the reply.

In [ ]:
from langgraph.graph import MessagesState
from langchain_core.prompts import (
    ChatPromptTemplate, SystemMessagePromptTemplate,
    HumanMessagePromptTemplate, MessagesPlaceholder,
)

prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(
        "You are a friendly museum guide. Reply briefly."
    ),
    MessagesPlaceholder("messages"),
])

def chatbot(state: MessagesState):
    messages = prompt.format_messages(messages=state["messages"])
    reply = llm.invoke(messages)
    return {"messages": [reply]}

builder = StateGraph(MessagesState)
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

Compile without a checkpointer and make two separate calls.

In [ ]:
graph = builder.compile()

graph.invoke({"messages": [HumanMessage("My name is Lea. My favourite colour is green.")]})["messages"][-1].pretty_print()

In [ ]:
graph.invoke({"messages": [HumanMessage("What is my name and my favourite colour?")]})["messages"][-1].pretty_print()

Each `invoke` receives only the new question here. Without a checkpointer, earlier calls are not restored.

## 3. Checkpointer and threads
A **checkpointer** saves the state after each step. A **thread** identifies one conversation: calls with the same `thread_id` continue the same saved state.

`InMemorySaver` keeps checkpoints in memory; they are lost when the kernel or server restarts.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

saver = InMemorySaver()
memory_graph = builder.compile(checkpointer=saver)
lea = {"configurable": {"thread_id": "lea"}}

memory_graph.invoke({"messages": [HumanMessage("My name is Lea. My favourite colour is green.")]}, lea)["messages"][-1].pretty_print()

In [ ]:
memory_graph.invoke({"messages": [HumanMessage("What is my name and my favourite colour?")]}, lea)["messages"][-1].pretty_print()

We sent only the new message. The checkpointer restored the earlier messages of thread `lea`. Inspect the saved state:

In [ ]:
saved = memory_graph.get_state(lea)
for message in saved.values["messages"]:
    message.pretty_print()

A different `thread_id` is a separate conversation.

In [ ]:
marco = {"configurable": {"thread_id": "marco"}}
memory_graph.invoke({"messages": [HumanMessage("What is my name?")]}, marco)["messages"][-1].pretty_print()

### Delete one conversation

Delete Lea's conversation after the Marco example above. Marco's thread remains. A new call with `lea` starts from an empty history.

In [ ]:
saver.delete_thread("lea")
print(memory_graph.get_state(lea).values)  # empty state

result = memory_graph.invoke(
    {"messages": [HumanMessage("What is my name?")]}, lea
)
result["messages"][-1].pretty_print()

## 4. A rolling buffer
The saved history grows with every turn. A rolling buffer sends only the most recent messages to the model; the full history stays in state.

In [ ]:
MAX_MESSAGES = 6

def buffered_chatbot(state: MessagesState):
    recent = state["messages"][-MAX_MESSAGES:]
    messages = prompt.format_messages(messages=recent)
    reply = llm.invoke(messages)
    return {"messages": [reply]}

builder = StateGraph(MessagesState)
builder.add_node("chatbot", buffered_chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)
buffered_graph = builder.compile(checkpointer=InMemorySaver())

### Your task
Use `buffered_graph` for two visitors.

1. Create configurations with `thread_id="lea"` and `thread_id="marco"`.
2. Tell the guide that Lea likes green and Marco likes blue, alternating threads. Ask each colour back.
3. Set `MAX_MESSAGES = 2`, rebuild the buffered graph and introduce the visitors again.
4. Send an unrelated request such as "Reply only OK", then ask the colour. Compare `buffered_graph.get_state(lea).values["messages"]` with the last `MAX_MESSAGES` messages sent to the model.

A reply can repeat the colour and keep it in the recent context. Check which messages the model actually sees; the complete conversation remains saved.

In [ ]:
# TODO: one thread per visitor, alternate the calls, then compare the replies.

In Chapter 03 the same graph runs inside the cognitive server: the avatar keeps one conversation thread, configured in the backend.

References: [Persistence](https://docs.langchain.com/oss/python/langgraph/persistence), [Graph API: reducers and MessagesState](https://docs.langchain.com/oss/python/langgraph/graph-api).